# Explainable Suspicious Transaction Detector — training notebook

Trains a **CatBoost** classifier (tuned with Optuna, calibrated) and an **Isolation Forest** anomaly signal on
[`nafiulislam490/bank-transaction-fraud-detection-dataset`](https://www.kaggle.com/datasets/nafiulislam490/bank-transaction-fraud-detection-dataset),
using **every data file** in the dataset, then exports `model_bundle.zip` for the local API.

The dataset is **synthetic**. Scores are "suspicious / flagged for review", never proof of fraud.

**Before running:** Runtime → Change runtime type → **T4 GPU** (CPU also works, just slower). Then Runtime → Run all.

## 1. Setup

In [ ]:
!pip install -q catboost optuna

In [ ]:
import os, re, glob, json, time, shutil, subprocess, warnings
import numpy as np, pandas as pd
import catboost, sklearn, optuna
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import (average_precision_score, roc_auc_score, precision_score, recall_score,
                             f1_score, confusion_matrix, precision_recall_curve, brier_score_loss)
from sklearn.isotonic import IsotonicRegression
from sklearn.ensemble import IsolationForest
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ---- settings -------------------------------------------------------------
SEED           = 42
DATA_DIR       = "data"
KAGGLE_SLUG    = "nafiulislam490/bank-transaction-fraud-detection-dataset"
LABEL          = "is_fraud"
EVAL_ONLY_COLS = ["fraud_type"]          # used for evaluation/audit only, NEVER as features
N_TRIALS       = 25                      # Optuna trials
TUNE_TIMEOUT_S = 25 * 60                 # hard cap on tuning time
TUNE_MAX_ROWS  = 300_000                 # tuning uses a sample of train (final model uses all of train)
THRESHOLD_MODE = "f1"                    # "f1" = best F1 on validation, or "budget"
ALERT_BUDGET   = 20                      # alerts per 1,000 txns, used when THRESHOLD_MODE == "budget"
# --------------------------------------------------------------------------

np.random.seed(SEED)
try:
    GPU = subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
except FileNotFoundError:
    GPU = False
print(f"catboost {catboost.__version__} | scikit-learn {sklearn.__version__} | pandas {pd.__version__} | GPU: {GPU}")

## 2. Download the dataset (all files)

In [ ]:
os.makedirs(DATA_DIR, exist_ok=True)
if not glob.glob(f"{DATA_DIR}/**/*.csv", recursive=True):
    kj = os.path.expanduser("~/.kaggle/kaggle.json")
    if not os.path.exists(kj) and not os.environ.get("KAGGLE_API_TOKEN"):
        from google.colab import files
        print("Upload kaggle.json  (kaggle.com → Settings → API → Create New Token)")
        files.upload()
        os.makedirs(os.path.dirname(kj), exist_ok=True)
        shutil.move("kaggle.json", kj)
        os.chmod(kj, 0o600)
    !kaggle datasets download -d {KAGGLE_SLUG} -p {DATA_DIR} --unzip

for f in sorted(glob.glob(f"{DATA_DIR}/**/*", recursive=True)):
    if os.path.isfile(f):
        print(f"{os.path.getsize(f)/1e6:9.1f} MB  {f}")

## 3. Load and merge every file

* Files with an `is_fraud` column and the same columns are **stacked**.
* Overlaps between files are detected: identical rows are removed; if IDs repeat with *different* content,
  the IDs are treated as file-local and prefixed with the file name.
* Transactions whose duplicates disagree on the label are dropped (reported).
* Unlabelled files with a unique key shared with the transactions (e.g. `customer_id`) are **joined** as lookup
  tables. Columns whose names suggest a label (`fraud`, `label`, `target`, …) are never joined.

In [ ]:
def norm_cols(d):
    d.columns = (d.columns.astype(str).str.strip().str.lower()
                 .str.replace(r"[^0-9a-z]+", "_", regex=True).str.strip("_"))
    return d

def read_any(f):
    if f.endswith(".csv"):              return pd.read_csv(f, low_memory=False)
    if f.endswith(".parquet"):          return pd.read_parquet(f)
    if f.endswith((".xlsx", ".xls")):   return pd.read_excel(f)
    if f.endswith(".json"):
        try: return pd.read_json(f)
        except ValueError: return pd.read_json(f, lines=True)
    return None

paths = sorted(f for f in glob.glob(f"{DATA_DIR}/**/*", recursive=True)
               if os.path.isfile(f) and f.lower().endswith((".csv", ".parquet", ".xlsx", ".xls", ".json")))
tables = {}
for f in paths:
    try:
        d = read_any(f)
    except Exception as e:
        print(f"could not read {f}: {e}"); continue
    if d is None or d.empty: continue
    tables[f] = norm_cols(d)
    print(f"{f}: {d.shape[0]:,} rows x {d.shape[1]} cols | label: {LABEL in d.columns}")

labelled = {f: d for f, d in tables.items() if LABEL in d.columns}
assert labelled, f"No file has a '{LABEL}' column"
main_f   = max(labelled, key=lambda f: len(labelled[f]))
BASE     = list(labelled[main_f].columns)
KEY      = "transaction_id" if "transaction_id" in BASE else None

parts, merge_log = [], []
for f, d in labelled.items():
    miss = set(BASE) - set(d.columns)
    if miss:
        merge_log.append(f"skipped {f}: missing {sorted(miss)}"); continue
    parts.append(d[BASE].assign(_source=os.path.basename(f)))
    merge_log.append(f"stacked {f}: {len(d):,} rows")
raw = pd.concat(parts, ignore_index=True)
n_stacked = len(raw)

if KEY and len(parts) > 1:
    dup = raw[raw.duplicated(KEY, keep=False)]
    if len(dup):
        same = dup.drop(columns="_source").duplicated(keep=False).mean()
        if same < 0.5:
            raw[KEY] = raw["_source"] + ":" + raw[KEY].astype(str)
            merge_log.append(f"{KEY} repeats across files with different content → IDs prefixed with file name")

raw = raw.loc[~raw.drop(columns="_source").duplicated()]            # exact duplicate rows
n_exact = n_stacked - len(raw)
n_conflict = 0
if KEY:
    nlab = raw.groupby(KEY)[LABEL].transform("nunique")
    n_conflict = int(raw.loc[nlab > 1, KEY].nunique())
    raw = raw.loc[nlab == 1].drop_duplicates(KEY, keep="first")
merge_log.append(f"removed {n_exact:,} exact duplicates, {n_conflict:,} IDs with conflicting labels")

for f, d in tables.items():
    if f in labelled: continue
    keys = [k for k in ("customer_id", "account_id", "merchant_id", "device_id") if k in d.columns and k in raw.columns]
    if not keys:
        merge_log.append(f"not used {f}: no shared key with transactions"); continue
    k = keys[0]
    if d[k].duplicated().any():
        merge_log.append(f"not used {f}: '{k}' is not unique, so it is not a lookup table"); continue
    new = [c for c in d.columns if c != k and c not in raw.columns
           and not re.search(r"fraud|label|target|class|chargeback|flag", c)]
    if not new:
        merge_log.append(f"not used {f}: no new columns"); continue
    raw[k] = raw[k].astype(str); d = d.copy(); d[k] = d[k].astype(str)
    raw = raw.merge(d[[k] + new], on=k, how="left")
    merge_log.append(f"joined {f} on {k}: +{new} (assumed known before each transaction)")

print("\n".join(merge_log))
print(f"\nfinal: {len(raw):,} rows x {raw.shape[1]} cols | fraud rate {raw[LABEL].mean():.4f}")
raw[LABEL] = raw[LABEL].astype(int)

## 4. Timestamp and ordering (needed for a time-based split)

In [ ]:
def build_ts(d):
    if {"transaction_date", "transaction_time"} <= set(d.columns):
        ts = pd.to_datetime(d["transaction_date"].astype(str) + " " + d["transaction_time"].astype(str), errors="coerce")
        if ts.isna().mean() < 0.5:
            return ts
    for c in ("timestamp", "transaction_datetime", "datetime", "transaction_date", "date", "time"):
        if c in d.columns:
            ts = pd.to_datetime(d[c], errors="coerce")
            if ts.isna().mean() < 0.5:
                return ts
    raise ValueError("No usable timestamp column found")

raw["ts"] = build_ts(raw)
bad = raw["ts"].isna().sum()
raw = raw.dropna(subset=["ts"])
sort_cols = ["ts"] + ([KEY] if KEY else [])
raw = raw.sort_values(sort_cols, kind="mergesort").reset_index(drop=True)   # ties broken by transaction_id
print(f"dropped {bad:,} rows without a timestamp | range {raw.ts.min()} → {raw.ts.max()}")

## 5. Data checks: fraud types, history per customer, single-feature leakage

In [ ]:
print("fraud rate:", round(raw[LABEL].mean(), 4))
for c in EVAL_ONLY_COLS:
    if c in raw: print(raw[c].value_counts(dropna=False).head(15), "\n")
if "customer_id" in raw:
    per = raw.groupby("customer_id").size()
    print(f"customers: {per.size:,} | txns/customer: median {per.median():.0f}, mean {per.mean():.1f}, max {per.max()}")

ID_LIKE = {"transaction_id", "customer_id", "account_id", "merchant_id", "device_id", "_source", "ts",
           "transaction_date", "transaction_time", LABEL, *EVAL_ONLY_COLS}
aucs = {}
for c in raw.columns:
    if c in ID_LIKE: continue
    s = raw[c]
    if pd.api.types.is_numeric_dtype(s) or pd.api.types.is_bool_dtype(s):
        x = s.astype(float).fillna(-1)
    else:
        x = s.astype(str).map(raw.groupby(s.astype(str))[LABEL].mean())      # in-sample rate: optimistic, a flag only
    if x.nunique() > 1:
        a = roc_auc_score(raw[LABEL], x); aucs[c] = max(a, 1 - a)
leak = pd.Series(aucs).sort_values(ascending=False)
print("\nsingle-feature AUC (top 12):\n", leak.head(12).round(3).to_string())
SUSPECT = leak[leak >= 0.97].index.tolist()
print("\n⚠ near-perfect single features (report these as a synthetic-data artefact):" if SUSPECT else "\nno single feature is near-perfect", SUSPECT or "")

## 6. Feature engineering (point-in-time)

Adds per-customer history features that use **only earlier transactions** of the same customer
(strictly earlier timestamp; same-timestamp rows are excluded), so nothing from the future leaks in.

In [ ]:
df = raw.copy()
AMT = next((c for c in ("transaction_amount", "amount") if c in df), None)
BAL = next((c for c in ("account_balance", "balance") if c in df), None)

if AMT:
    df["log_amount"] = np.log1p(df[AMT].clip(lower=0))
    if BAL:
        df["amount_to_balance"] = df[AMT] / (df[BAL].abs() + 1)

HIST_FEATS = []
if "customer_id" in df and AMT:
    df["_cust"] = pd.factorize(df["customer_id"].astype(str))[0].astype(np.int64)
    order = np.lexsort((np.arange(len(df)), df["ts"].values, df["_cust"].values))     # customer, time, row
    d = df.iloc[order]
    t = (d["ts"].values.astype("datetime64[s]").astype(np.int64))
    t = t - t.min()
    span = int(t.max()) + 10 * 86400
    k = d["_cust"].values * span + t                                                   # monotonic key
    amt = d[AMT].values.astype(float)
    right = np.searchsorted(k, k, side="left")                                         # strictly before this timestamp
    first = np.searchsorted(k, d["_cust"].values * span, side="left")                  # customer's first row
    cust_arr = d["_cust"].values
    # Running totals are kept WITHIN each customer (not across the whole file), so they stay
    # numerically exact on 1M rows and give identical results when future rows are added.
    gcum = lambda x: pd.Series(x).groupby(cust_arr).cumsum().values
    def range_sum(gc, lo, hi):                                                         # sum of this customer's rows lo..hi-1
        top = np.where(hi > first, gc[np.maximum(hi - 1, 0)], 0.0)
        bot = np.where(lo > first, gc[np.maximum(lo - 1, 0)], 0.0)
        return top - bot
    g1 = gcum(amt)
    out = pd.DataFrame(index=d.index)
    n_prior = right - first
    out["cust_prior_txns"] = n_prior
    prior_mean = np.where(n_prior > 0, range_sum(g1, first, right) / np.maximum(n_prior, 1), np.nan)
    out["amt_vs_cust_mean"] = np.where(n_prior > 0, amt / (prior_mean + 1), np.nan)
    for name, w in (("1h", 3600), ("24h", 86400), ("7d", 7 * 86400)):
        left = np.searchsorted(k, k - w, side="left")
        left = np.maximum(left, first)
        out[f"cust_txns_{name}"] = right - left
        out[f"cust_amt_{name}"] = range_sum(g1, left, right)
    prev_t = np.where(n_prior > 0, t[np.maximum(right - 1, 0)], -1)
    out["hrs_since_cust_prev"] = np.where(n_prior > 0, (t - prev_t) / 3600, np.nan)

    # --- per-customer baselines: is this unusual FOR THIS customer? (strictly earlier txns only) ---
    def prior_same(group):
        codes = d["_cust"].values * 32 + group
        o = np.lexsort((t, codes)); kk = codes[o] * span + t[o]
        c = np.searchsorted(kk, kk, side="left") - np.searchsorted(kk, codes[o] * span, side="left")
        res = np.empty_like(c); res[o] = c
        return res
    hr, dow = d["ts"].dt.hour.values, d["ts"].dt.dayofweek.values
    band = np.select([hr < 6, hr < 12, hr < 18], [0, 1, 2], 3)          # night / morning / afternoon / evening
    enough = n_prior >= 5                                              # below this, the baseline is too thin
    denom = np.maximum(n_prior, 1)
    band_share = np.where(n_prior > 0, prior_same(band) / denom, np.nan)
    day_share = np.where(n_prior > 0, prior_same(dow) / denom, np.nan)
    is_night = (hr < 6).astype(int)
    same_night = prior_same(is_night)
    out["hour_band_share_cust"] = band_share
    out["unusual_hour_for_cust"] = np.where(enough, (band_share < 0.10).astype(int), -1)
    out["night_share_cust"] = np.where(n_prior > 0, np.where(is_night == 1, same_night, n_prior - same_night) / denom, np.nan)
    out["weekday_share_cust"] = day_share
    out["unusual_day_for_cust"] = np.where(enough, (day_share < 0.05).astype(int), -1)
    a0 = amt - amt[first]                                   # shift by customer's first amount: variance is unchanged, precision is kept
    s1, s2 = range_sum(gcum(a0), first, right), range_sum(gcum(a0 ** 2), first, right)
    var = np.where(n_prior >= 2, (s2 - s1 ** 2 / denom) / np.maximum(n_prior - 1, 1), np.nan)
    out["amt_z_cust"] = np.where(n_prior >= 2, (amt - prior_mean) / (np.sqrt(np.clip(var, 0, None)) + 1), np.nan)
    df = df.join(out)
    for c in ("city", "country", "device_type", "payment_method", "merchant_category"):
        if c in df:
            seen = df.sort_values(["_cust", "ts"], kind="mergesort").duplicated(["_cust", c])
            df[f"new_{c}_for_cust"] = np.where(df["cust_prior_txns"] > 0, (~seen.reindex(df.index)).astype(int), -1)
    df["limited_history"] = (df["cust_prior_txns"] < 3).astype(int)
    HIST_FEATS = [c for c in out.columns] + [c for c in df.columns if c.startswith("new_") and c.endswith("_for_cust")] + ["limited_history"]
    df = df.drop(columns="_cust")
    print(f"history features: {len(HIST_FEATS)} | rows with ≥1 prior txn: {(df.cust_prior_txns > 0).mean():.1%}")

EXCLUDE = ID_LIKE | {c for c in df.columns if "fraud" in c or c in ("label", "target", "class")}
FEATS = [c for c in df.columns if c not in EXCLUDE and not pd.api.types.is_datetime64_any_dtype(df[c])]
for c in FEATS:
    if pd.api.types.is_bool_dtype(df[c]): df[c] = df[c].astype(int)
CATS = [c for c in FEATS if not pd.api.types.is_numeric_dtype(df[c])]
NUMS = [c for c in FEATS if c not in CATS]
for c in CATS: df[c] = df[c].astype(str).fillna("NA")

assert LABEL not in FEATS and not set(EVAL_ONLY_COLS) & set(FEATS) and not {"transaction_id", "customer_id"} & set(FEATS)
print(f"{len(FEATS)} features | categorical: {CATS}")

## 7. Time-based split 70 / 15 / 15 (no shuffling)

In [ ]:
n = len(df); a, b = int(n * 0.70), int(n * 0.85)
train, val, test = df.iloc[:a].copy(), df.iloc[a:b].copy(), df.iloc[b:].copy()
assert train.ts.max() <= val.ts.min() and val.ts.max() <= test.ts.min(), "time overlap between splits"
for name, s in (("train", train), ("val", val), ("test", test)):
    print(f"{name:5s} {len(s):>9,} rows | {s.ts.min()} → {s.ts.max()} | fraud rate {s[LABEL].mean():.4f}")

def pool(d, labelled=True):
    return Pool(d[FEATS], d[LABEL] if labelled else None, cat_features=CATS)
vpool = pool(val)

## 8. Hyperparameter tuning (Optuna, scored on validation PR-AUC)

In [ ]:
tune_tr = train.sample(min(TUNE_MAX_ROWS, len(train)), random_state=SEED)
tpool_small = pool(tune_tr)

def fit_cb(params, tp, vp, iters, es):
    base = dict(iterations=iters, eval_metric="PRAUC", early_stopping_rounds=es, random_seed=SEED,
                verbose=0, allow_writing_files=False)
    if GPU:
        try:
            return CatBoostClassifier(**base, **params, task_type="GPU").fit(tp, eval_set=vp)
        except Exception as e:
            print("GPU fit failed, using CPU:", str(e)[:120])
    return CatBoostClassifier(**base, **params, thread_count=-1).fit(tp, eval_set=vp)

def objective(trial):
    p = dict(
        depth=trial.suggest_int("depth", 4, 9),
        learning_rate=trial.suggest_float("learning_rate", 0.02, 0.25, log=True),
        l2_leaf_reg=trial.suggest_float("l2_leaf_reg", 1.0, 20.0, log=True),
        random_strength=trial.suggest_float("random_strength", 0.0, 3.0),
        bagging_temperature=trial.suggest_float("bagging_temperature", 0.0, 1.0),
        border_count=trial.suggest_categorical("border_count", [128, 254]),
        auto_class_weights=trial.suggest_categorical("auto_class_weights", ["None", "Balanced", "SqrtBalanced"]),
    )
    if p["auto_class_weights"] == "None": p.pop("auto_class_weights")
    m = fit_cb(p, tpool_small, vpool, iters=1500, es=100)
    trial.set_user_attr("best_iteration", m.get_best_iteration())
    return average_precision_score(val[LABEL], m.predict_proba(vpool)[:, 1])

t0 = time.time()
study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
study.enqueue_trial(dict(depth=6, learning_rate=0.08, l2_leaf_reg=3.0, random_strength=1.0,
                         bagging_temperature=1.0, border_count=254, auto_class_weights="Balanced"))
study.optimize(objective, n_trials=N_TRIALS, timeout=TUNE_TIMEOUT_S, show_progress_bar=True)
BEST = dict(study.best_params)
if BEST.get("auto_class_weights") == "None": BEST.pop("auto_class_weights")
print(f"{len(study.trials)} trials in {time.time()-t0:.0f}s | best validation PR-AUC {study.best_value:.4f}\n{BEST}")

## 9. Final CatBoost model (all of train, early stopping on validation)

In [ ]:
t0 = time.time()
model = fit_cb(BEST, pool(train), vpool, iters=4000, es=200)
print(f"trained in {time.time()-t0:.0f}s | best iteration {model.get_best_iteration()}")
raw_val  = model.predict_proba(vpool)[:, 1]
raw_test = model.predict_proba(pool(test))[:, 1]
print("validation PR-AUC (uncalibrated):", round(average_precision_score(val[LABEL], raw_val), 4))

## 10. Calibration (isotonic, fitted on validation) and alert threshold (chosen on validation)

In [ ]:
iso = IsotonicRegression(out_of_bounds="clip", y_min=0, y_max=1).fit(raw_val, val[LABEL])
calibrate = lambda p: np.interp(p, iso.X_thresholds_, iso.y_thresholds_)
p_val, p_test = calibrate(raw_val), calibrate(raw_test)

def pick_threshold(y, p):
    if THRESHOLD_MODE == "budget":
        return float(np.quantile(p, 1 - ALERT_BUDGET / 1000))
    P, R, T = precision_recall_curve(y, p)
    f1 = 2 * P[:-1] * R[:-1] / (P[:-1] + R[:-1] + 1e-12)
    return float(T[np.nanargmax(f1)])

THR = pick_threshold(val[LABEL], p_val)
print(f"alert threshold ({THRESHOLD_MODE}): {THR:.4f}")

## 11. Isolation Forest (unsupervised anomaly signal — not a probability)

In [ ]:
FILL = train[NUMS].median(numeric_only=True).to_dict()
Xif = lambda d: d[NUMS].astype(float).fillna(FILL).values
ifo = IsolationForest(n_estimators=300, max_samples=256, random_state=SEED, n_jobs=-1)
ifo.fit(Xif(train.sample(min(300_000, len(train)), random_state=SEED)))       # labels are NOT used
IF_Q = np.quantile(-ifo.score_samples(Xif(train.sample(min(200_000, len(train)), random_state=SEED))),
                   np.linspace(0, 1, 1001))
anom_pct = lambda d: np.minimum(np.searchsorted(IF_Q, -ifo.score_samples(Xif(d))) / 10.0, 100.0)   # 0–100 percentile vs train
a_val, a_test = anom_pct(val), anom_pct(test)
print("anomaly percentile, test: mean", a_test.mean().round(1))

## 12. Evaluation on the untouched test split

In [ ]:
def metrics(y, score, thr):
    yhat = (score >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, yhat, labels=[0, 1]).ravel()
    return dict(avg_precision=round(float(average_precision_score(y, score)), 4),
                roc_auc=round(float(roc_auc_score(y, score)), 4),
                precision=round(float(precision_score(y, yhat, zero_division=0)), 4),
                recall=round(float(recall_score(y, yhat, zero_division=0)), 4),
                f1=round(float(f1_score(y, yhat, zero_division=0)), 4),
                alerts_per_1000=round(1000 * float(yhat.mean()), 2),
                confusion=dict(tn=int(tn), fp=int(fp), fn=int(fn), tp=int(tp)))

yv, yt = val[LABEL].values, test[LABEL].values
# Baseline: amount only, threshold chosen on validation
amt_thr = pick_threshold(yv, val[AMT].values) if AMT else None
# Isolation Forest: threshold set on validation at the same alert rate as CatBoost (fair comparison)
if_thr = float(np.quantile(a_val, 1 - (p_val >= THR).mean()))

EVAL = {"catboost": metrics(yt, p_test, THR),
        "isolation_forest": metrics(yt, a_test, if_thr)}
if AMT: EVAL["amount_threshold_baseline"] = metrics(yt, test[AMT].values, amt_thr)
print(pd.DataFrame({k: {m: v for m, v in d.items() if m != "confusion"} for k, d in EVAL.items()}).T.to_string())
print("\nCatBoost confusion (test):", EVAL["catboost"]["confusion"])

# Calibration on test
bins = pd.qcut(p_test, 10, duplicates="drop")
calib = pd.DataFrame({"p": p_test, "y": yt}).groupby(bins, observed=True).agg(predicted=("p", "mean"), observed=("y", "mean"), n=("y", "size"))
EVAL["calibration"] = dict(brier=round(float(brier_score_loss(yt, p_test)), 5),
                           reliability=calib.round(4).reset_index(drop=True).to_dict("records"))
print("\nBrier score:", EVAL["calibration"]["brier"]); print(calib.round(3).to_string())

test["score"], test["alert"], test["anomaly_pct"] = p_test, (p_test >= THR).astype(int), a_test

# Recall per fraud type
for c in EVAL_ONLY_COLS:
    if c in test:
        r = test[test[LABEL] == 1].groupby(c)["alert"].agg(["mean", "size"]).rename(columns={"mean": "recall", "size": "n"})
        EVAL[f"recall_by_{c}"] = r.round(4).reset_index().to_dict("records"); print(f"\nrecall by {c}:\n", r.round(3).to_string())

# Alert-rate check across groups (possible bias)
groups = [c for c in ("country", "device_type", "payment_method", "merchant_category") if c in test]
if "customer_age" in test:
    test["age_band"] = pd.cut(test["customer_age"], [0, 25, 35, 50, 65, 200], labels=["<25", "25-34", "35-49", "50-64", "65+"]).astype(str)
    groups.append("age_band")
EVAL["group_check"] = {}
for g in groups:
    t = test.groupby(g).agg(n=(LABEL, "size"), alert_rate=("alert", "mean"), fraud_rate=(LABEL, "mean"))
    t["recall"] = test[test[LABEL] == 1].groupby(g)["alert"].mean()
    EVAL["group_check"][g] = t.round(4).reset_index().to_dict("records")
    print(f"\nby {g}:\n", t.round(3).to_string())

## 13. Explanations (CatBoost native SHAP), explanation audit, false-positive example, "What changed?"

SHAP values are in **log-odds** units. Reason families are assigned from feature names (printed below — check them).

In [ ]:
FAMILY_RULES = [("velocity", ["freq", "prev", "since", "last", "txns_", "prior", "_1h", "_24h", "_7d"]),
                ("amount",   ["amount", "balance", "amt"]),
                ("security", ["failed", "pin", "attempt", "login", "device"]),
                ("location", ["distance", "international", "country", "city", "home"]),
                ("time",     ["hour", "night", "weekend", "day"])]
def family(f):
    for fam, keys in FAMILY_RULES:
        if any(k in f for k in keys): return fam
    return "profile"
FAMILY = {f: family(f) for f in FEATS}
print(pd.Series(FAMILY).sort_values().to_string())

BASELINE = {**{c: float(train[c].median()) for c in NUMS}, **{c: str(train[c].mode().iloc[0]) for c in CATS}}

def shap_rows(d):
    sv = model.get_feature_importance(Pool(d[FEATS], cat_features=CATS), type="ShapValues")
    return sv[:, :-1]

def fmt(v):
    if isinstance(v, (float, np.floating)):
        return f"{v:,.2f}" if abs(v) < 1000 else f"{v:,.0f}"
    return str(v)

def as_frame(row):
    r = row.to_frame().T
    for c in NUMS: r[c] = pd.to_numeric(r[c], errors="coerce").astype(float)
    for c in CATS: r[c] = r[c].astype(str)
    return r

def reason_text(f, x):
    # Plain-English reason; every number comes from the row itself or the training baseline.
    v = x[f]
    if pd.isna(v): return None
    if f == "unusual_hour_for_cust":
        return f"unusual time of day for this customer: {x['hour_band_share_cust']:.0%} of their earlier transactions were in this time band" if v == 1 else None
    if f == "hour_band_share_cust":
        return f"{v:.0%} of this customer's earlier transactions were at this time of day"
    if f == "unusual_day_for_cust":
        return f"unusual day for this customer: {x['weekday_share_cust']:.0%} of their earlier transactions were on this weekday" if v == 1 else None
    if f == "weekday_share_cust":
        return f"{v:.0%} of this customer's earlier transactions were on this weekday"
    if f == "night_share_cust":
        return (f"night-time transaction; {v:.0%} of this customer's earlier transactions were at night"
                if "is_night_transaction" in x and x["is_night_transaction"] == 1 else None)
    if f == "is_night_transaction":
        return "made at night" if v == 1 else None
    if f == "amt_vs_cust_mean":
        return f"amount is {v:.1f}x this customer's usual amount"
    if f == "amt_z_cust":
        return f"amount is {v:.1f} standard deviations from this customer's usual amount"
    if f.startswith("new_") and f.endswith("_for_cust"):
        return f"first transaction with this {f[4:-9].replace('_', ' ')} for this customer" if v == 1 else None
    if f.startswith("cust_txns_"):
        return f"{int(v)} earlier transactions by this customer in the past {f.split('_')[-1]}"
    if f == "hrs_since_cust_prev":
        return f"{v:.2f} hours since this customer's previous transaction"
    if f == "failed_attempts":
        return f"{int(v)} failed attempts before this transaction"
    if f == "pin_changed_recently":
        return "PIN changed recently" if v == 1 else None
    if f == "distance_from_home_km":
        return f"{v:,.0f} km from home (typical {fmt(BASELINE[f])} km)"
    if f == "limited_history":
        return None
    return f"{f.replace('_', ' ')} = {fmt(v)} (typical {fmt(BASELINE[f])})"

def explain(row, top=5):
    r = as_frame(row)
    sv = shap_rows(r)[0]
    x = r.iloc[0]
    order = np.argsort(-np.abs(sv))
    reasons = []
    for i in order:
        if sv[i] <= 0 or len(reasons) >= top: continue
        txt = reason_text(FEATS[i], x)
        if txt:                                   # only mention a reason the evidence supports
            reasons.append(dict(feature=FEATS[i], value=fmt(x[FEATS[i]]), shap=round(float(sv[i]), 4),
                                family=FAMILY[FEATS[i]], text=txt))
    fam = pd.Series(sv, index=FEATS).groupby(FAMILY).sum().sort_values(ascending=False).round(4).to_dict()
    score = float(calibrate(model.predict_proba(Pool(r[FEATS], cat_features=CATS))[:, 1])[0])
    text = ("Flagged for review" if score >= THR else "Not flagged") + f" (score {score:.2f}). " + \
           ("Main factors: " + "; ".join(z["text"] for z in reasons[:3]) + "."
            if reasons else "No factor pushed the score up.")
    if "limited_history" in r and int(r.iloc[0]["limited_history"]) == 1:
        text += " Limited history for this customer: treat the comparison with caution."
    return dict(score=round(score, 4), reasons=reasons, families=fam, text=text)

def what_changed(row, fam):
    r = as_frame(row)
    for f in FEATS:
        if FAMILY[f] == fam: r[f] = BASELINE[f]
    new = float(calibrate(model.predict_proba(Pool(r[FEATS], cat_features=CATS))[:, 1])[0])
    return dict(family=fam, original=explain(row)["score"], hypothetical=round(new, 4),
                note="Model behaviour under a hypothetical change. This is not proof of causation.")

# Explanation audit: top reason family for true-positive alerts, by fraud type
tp = test[(test.alert == 1) & (test[LABEL] == 1)]
tp = tp.sample(min(3000, len(tp)), random_state=SEED)
if len(tp):
    sv = shap_rows(tp)
    tp_fam = pd.DataFrame(sv, columns=FEATS).T.groupby(FAMILY).sum().T.idxmax(axis=1).values
    tp = tp.assign(top_family=tp_fam)
    for c in EVAL_ONLY_COLS:
        if c in tp:
            audit = pd.crosstab(tp[c], tp.top_family, normalize="index").round(3)
            EVAL["explanation_audit"] = audit.reset_index().to_dict("records")
            print("\nTop reason family by fraud type (rows sum to 1):\n", audit.to_string())

# Global importance
imp = pd.Series(np.abs(shap_rows(test.sample(min(5000, len(test)), random_state=SEED))).mean(0), index=FEATS).sort_values(ascending=False)
EVAL["global_importance"] = imp.round(5).head(20).to_dict()
print("\nMean |SHAP| (top 15):\n", imp.head(15).round(4).to_string())

# Examples
top_alert = test[(test.alert == 1) & (test[LABEL] == 1)].sort_values("score").iloc[-1] if len(tp) else None
fp = test[(test.alert == 1) & (test[LABEL] == 0)].sort_values("score")
if top_alert is not None:
    e = explain(top_alert); print("\nALERT:", e["text"]); print(pd.DataFrame(e["reasons"]).to_string())
    print(what_changed(top_alert, max(e["families"], key=e["families"].get)))
if len(fp):
    e = explain(fp.iloc[-1]); EVAL["false_positive_example"] = dict(e, transaction_id=str(fp.iloc[-1].get(KEY, "")))
    print("\nFALSE POSITIVE (legitimate but flagged):", e["text"]); print(pd.DataFrame(e["reasons"]).to_string())

## 14. Export `model_bundle.zip` for the local API

In [ ]:
B = "model_bundle"; shutil.rmtree(B, ignore_errors=True); os.makedirs(B)
import joblib
model.save_model(f"{B}/catboost_model.cbm")
json.dump({"x": iso.X_thresholds_.tolist(), "y": iso.y_thresholds_.tolist()}, open(f"{B}/calibrator.json", "w"))
joblib.dump(ifo, f"{B}/iforest.joblib")
json.dump({"quantiles": IF_Q.tolist()}, open(f"{B}/iforest_quantiles.json", "w"))
config = dict(dataset=KAGGLE_SLUG, label=LABEL, features=FEATS, cat_features=CATS, num_features=NUMS,
              history_features=HIST_FEATS, if_fill=FILL, baseline=BASELINE, family=FAMILY,
              threshold=THR, threshold_mode=THRESHOLD_MODE, best_params=BEST,
              best_iteration=int(model.get_best_iteration()), seed=SEED, merge_log=merge_log,
              suspect_features=SUSPECT,
              split=dict(train=[str(train.ts.min()), str(train.ts.max())], val=[str(val.ts.min()), str(val.ts.max())],
                         test=[str(test.ts.min()), str(test.ts.max())]),
              versions=dict(catboost=catboost.__version__, scikit_learn=sklearn.__version__,
                            pandas=pd.__version__, numpy=np.__version__),
              notes="Synthetic dataset. Scores mean 'flagged for review', not proof of fraud. "
                    "Anomaly percentile is not a probability.")
json.dump(config, open(f"{B}/config.json", "w"), indent=2, default=str)
json.dump(EVAL, open(f"{B}/evaluation.json", "w"), indent=2, default=str)

keep = [c for c in [KEY, "customer_id", "ts", *FEATS, LABEL, *EVAL_ONLY_COLS, "score", "alert", "anomaly_pct"] if c and c in test]
test[keep].to_csv(f"{B}/demo_transactions.csv.gz", index=False, compression="gzip")
shutil.make_archive("model_bundle", "zip", B)
print(f"model_bundle.zip  {os.path.getsize('model_bundle.zip')/1e6:.1f} MB")
print("Install these versions locally:", config["versions"])

In [ ]:
from google.colab import files
files.download("model_bundle.zip")